#AI Fitness Assistant

In [1]:
!pip install -q openai chromadb pypdf pydantic python-dotenv pymupdf
!pip install -q langchain-text-splitters tiktoken

In [2]:
import os
import re
import json
from typing import List, Dict, Any, Iterable, Literal

import openai
from openai import OpenAI

from pydantic import BaseModel, Field
from dataclasses import dataclass
from __future__ import annotations
from collections import Counter
from pathlib import Path
import pymupdf

from google.colab import userdata
import chromadb
from chromadb.config import Settings
from chromadb.utils.embedding_functions import OpenAIEmbeddingFunction

import tiktoken
from langchain_text_splitters import RecursiveCharacterTextSplitter

from IPython.display import display, Markdown

Parameters

In [3]:

# Paths
PDF_PATH = "/content/fitness_book.pdf"
EXERCISE_JSON_PATH = "/content/fitness_data.json"
PATH_TO_AUDIO_OUTPUT = Path("/content/last_answer.mp3")

# Chroma collection
COLLECTION_NAME = "fitness_multisource_rag"

# Models
EMBEDDING_MODEL = "text-embedding-3-small"
MODEL = "gpt-5-nano"

# Chunking
CHUNK_SIZE = 800
CHUNK_OVERLAP = 100
TOP_K_RESULTS = 5


Init clients

In [31]:
api_key = userdata.get('OPENAI_API_KEY')

client = OpenAI(api_key=api_key)
conversation = client.conversations.create()



In [5]:

# Create a persistent Chroma database
PATH_TO_CHROMA_DB = "/content/chromadb"
chroma_client = chromadb.PersistentClient(path=PATH_TO_CHROMA_DB)


collection = chroma_client.get_or_create_collection(name="book_chunks", metadata={"hnsw:space": "cosine"}, embedding_function=OpenAIEmbeddingFunction(api_key=api_key, model_name="text-embedding-3-small"))
exercise_collection = chroma_client.get_or_create_collection(name="exercise_chunks")

In [33]:
class ExerciseRecord(BaseModel):
    exercise_name: str
    type_of_activity: str
    type_of_equipment: str
    body_part: str
    type: str
    muscle_groups_activated: str
    instructions: str


class RetrieveExercisesArgs(BaseModel):
    query: str = Field(..., description="The user's request for additional exercises.")
    top_k: int = Field(TOP_K_RESULTS, description="Number of relevant exercises to retrieve.")

class AskAIResponse(BaseModel):
    prompt: str
    format: Literal["text", "audio"]

Cleaning and loading pdf

In [7]:

def normalize_line(s: str) -> str:
    s = s.strip().replace("\u00A0", " ")
    s = re.sub(r"\s+", " ", s)
    s = re.sub(r"\b(?:\d+|[ivxlcdm]+)\b", "#", s, flags=re.I)
    return s.lower()


def is_junk_line(s: str) -> bool:
    s = s.strip()
    return (
        not s
        or bool(re.fullmatch(r"(?i)this page intentionally left blank\.?", s))
        or bool(re.fullmatch(r"\d{1,4}", s))
        or bool(re.fullmatch(r"(?i)[ivxlcdm]{1,12}", s))
        or bool(re.fullmatch(r"[A-Z]{1,3}\d{1,4}", s))
        or bool(re.fullmatch(r"[-–—_*~.=•·\s]{3,}", s))
    )


def iter_text_lines(page: pymupdf.Page):
    for block in page.get_text("dict", sort=True).get("blocks", []):
        if block.get("type") != 0:
            continue
        for line in block.get("lines", []):
            spans = line.get("spans", [])
            bbox = line.get("bbox")
            if not spans or not bbox:
                continue

            text = "".join(span.get("text", "") for span in spans).strip()
            if not text:
                continue

            yield text, bbox


def detect_repeated_margin_lines(
    doc: pymupdf.Document,
    sample_pages: int = 40,
    top_ratio: float = 0.10,
    bottom_ratio: float = 0.10,
    min_repeat_fraction: float = 0.20,
) -> set[str]:
    limit = min(len(doc), sample_pages)
    counter = Counter()

    for page in doc[:limit]:
        h = page.rect.height
        top_limit = h * top_ratio
        bottom_limit = h * (1.0 - bottom_ratio)

        seen = {
            normalize_line(text)
            for text, (_, y0, _, y1) in iter_text_lines(page)
            if y1 <= top_limit or y0 >= bottom_limit
        }

        for line in seen:
            if len(line) >= 5:
                counter[line] += 1

    threshold = max(3, int(limit * min_repeat_fraction))
    return {line for line, count in counter.items() if count >= threshold}


def clean_page_text(text: str) -> str:
    if not text:
        return ""

    substitutions = [
        (r"\r\n?", "\n"),
        (r"\u00A0|\t", " "),
        (r"[ \u2000-\u200B\u202F\u205F\u3000]+", " "),
        (r"(?im)^\s*this page intentionally left blank\.?\s*$", ""),
        (r"(?im)^\s*(?:\d+|[ivxlcdm]+)\s*$", ""),
        (r"(?m)^\s*[A-Z]{1,3}\d{1,4}\s*$", ""),
        (r"(?<=\w)-\n(?=\w)", ""),
        (r"\n(?=[a-z])", " "),
        (r"[ ]{2,}", " "),
        (r"\s+([,.;:!?])", r"\1"),
        (r"\n{3,}", "\n\n"),
    ]

    for pattern, repl in substitutions:
        text = re.sub(pattern, repl, text)

    return text.strip()


def extract_clean_pages(
    pdf_path: str | Path,
    top_crop_ratio: float = 0.08,
    bottom_crop_ratio: float = 0.08,
) -> list[dict]:
    doc = pymupdf.open(str(pdf_path))
    try:
        repeated = detect_repeated_margin_lines(doc) if len(doc) >= 5 else set()
        pages = []

        for page_num, page in enumerate(doc, start=1):
            h = page.rect.height
            top_cut = h * top_crop_ratio
            bottom_cut = h * (1.0 - bottom_crop_ratio)

            lines = [
                text
                for text, (_, y0, _, y1) in iter_text_lines(page)
                if not (y1 <= top_cut or y0 >= bottom_cut)
                and normalize_line(text) not in repeated
                and not is_junk_line(text)
            ]

            page_text = clean_page_text("\n".join(lines))
            if page_text:
                pages.append({"page": page_num, "text": page_text})

        return pages
    finally:
        doc.close()


def split_by_chapters(pages: list[dict]) -> list[dict]:
    chapter_pat = re.compile(r"(?im)^\s*chapter(?:\s+(\d+))?\s*$")
    part_pat = re.compile(r"(?im)^\s*part\s+([ivxlcdm]+|\d+)(?::.*)?\s*$")

    def build_chapter_title(number: int | None, next_line: str | None) -> str:
        base = f"Chapter {number}" if number is not None else "Chapter"
        if (
            next_line
            and len(next_line) <= 100
            and not chapter_pat.fullmatch(next_line)
            and not part_pat.fullmatch(next_line)
            and not next_line.endswith((".", "?", "!"))
        ):
            return f"{base}: {next_line}"
        return base

    chapters = []
    current = None
    current_part = None

    for page in pages:
        text = page["text"]

        part_match = part_pat.search(text)
        if part_match:
            current_part = part_match.group(0).strip()

        lines = [line.strip() for line in text.split("\n") if line.strip()]
        chapter_idx = None
        chapter_number = None
        chapter_title = None

        for i, line in enumerate(lines):
            match = chapter_pat.fullmatch(line)
            if not match:
                continue

            chapter_idx = i
            chapter_number = int(match.group(1)) if match.group(1) else None
            next_line = lines[i + 1] if i + 1 < len(lines) else None
            chapter_title = build_chapter_title(chapter_number, next_line)
            break

        if chapter_idx is not None:
            if current:
                chapters.append(current)

            current = {
                "part": current_part,
                "chapter_number": chapter_number,
                "chapter_title": chapter_title,
                "start_page": page["page"],
                "pages": [page["page"]],
                "text": text,
            }
        elif current:
            current["pages"].append(page["page"])
            current["text"] += "\n\n" + text

    if current:
        chapters.append(current)

    return chapters

In [8]:
pages = extract_clean_pages(PDF_PATH)
chapters = split_by_chapters(pages)


In [ ]:

print(f"Pages: {len(pages)}")
print(f"Chapters: {len(chapters)}")
print()

for i, ch in enumerate(chapters, start=1):
    print("=" * 120)
    print(f"index: {i}")
    print(f"chapter_number: {ch.get('chapter_number')}")
    print(f"chapter_title: {ch.get('chapter_title')}")
    print(f"pages: {ch.get('start_page')} -> {ch.get('pages', [None])[-1]}")
    print(f"text_len: {len(ch.get('text', ''))}")
    print("-" * 120)
    print(ch.get("text", "")[:150000])   # preview only
    print()

Pages: 264
Chapters: 17

index: 1
chapter_number: 1
chapter_title: Chapter 1: Programmed Response
pages: 7 -> 7
text_len: 320
------------------------------------------------------------------------------------------------------------------------
Contents
Preface ix
Acknowledgments xiii
PART I Programming: Going Beyond the Basics
CHAPTER 1
Programmed Response
CHAPTER 2
Accurately Assessing Current
Training Status
CHAPTER 3
Progressing and Regressing Exercises on the Fly
PART II The Training Programs
CHAPTER 4
Fat-Loss Programs
CHAPTER 5
Muscle-Building Programs

index: 2
chapter_number: 6
chapter_title: Chapter 6: Strength-Building Programs
pages: 8 -> 16
text_len: 12613
------------------------------------------------------------------------------------------------------------------------
CHAPTER 6
Strength-Building Programs
CHAPTER 7
General Fitness Programs
CHAPTER 8
Semiprivate and Group Training Programs 163
CHAPTER 9
The Fitness Coach’s Toolbox of Workouts 203
PART III Evaluation

Chunking the main document

In [9]:

enc = tiktoken.get_encoding("cl100k_base")

def token_len(text: str) -> int:
    return len(enc.encode(text, disallowed_special=()))



In [10]:
def split_chapters_into_chunks(chapters, chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=token_len,
        separators=["\n\n", "\n", ". ", " ", ""],
    )

    excluded_indexes = {1, 2, 14, 15, 16, 17}
    chunks = []

    for ch_idx, ch in enumerate(chapters, start=1):
        if ch_idx in excluded_indexes:
            continue

        chapter_text = ch["text"].strip()
        if not chapter_text:
            continue

        parts = splitter.split_text(chapter_text)

        actual_ch_num = ch.get("chapter_number")
        chapter_id = f"{actual_ch_num:02d}" if isinstance(actual_ch_num, int) else f"x{ch_idx:02d}"

        for part_idx, part in enumerate(parts, start=1):
            chunks.append({
                "chunk_id": f"ch{chapter_id}-{part_idx:03d}",
                "chapter_number": ch.get("chapter_number"),
                "chapter_title": ch.get("chapter_title"),
                "token_count": token_len(part),
                "text": part.strip(),
            })

    return chunks

In [11]:
chunks = split_chapters_into_chunks(chapters, chunk_size=800, chunk_overlap=100)

print("Total chunks:", len(chunks))
print()

# Show first 5 full chunks
for chunk in chunks[:5]:
    print("=" * 120)
    print("chunk_id:", chunk["chunk_id"])
    print("chapter_number:", chunk["chapter_number"])
    print("chapter_title:", chunk["chapter_title"])
    print("token_count:", chunk["token_count"])
    print("-" * 120)
    print(chunk["text"])
    print()

Total chunks: 181

chunk_id: chx03-001
chapter_number: None
chapter_title: Chapter: Programmed Response
token_count: 616
------------------------------------------------------------------------------------------------------------------------
CHAPTER
Programmed Response
The moment Alwyn became self-actualized with regard to programming was when he realized that he could make an adaptation happen by directing the process physiologically, an approach otherwise known as, “results by design, not by coincidence.” Specific and intentional results could be part of the plan, which is far different than simply doing a random training session and waiting to see what might happen. The process of imposing an appropriate stressor and expecting an adaption could be demonstrable and repeatable; this is the SAID (Specific Adaptation to Imposed Demands) principle in action. Training, in a nutshell, is simply imposing an appropriate stressor on the body, and the body adapting to it during the recovery pe

Storing the main document in DB

In [12]:

collection.upsert(
    ids=[chunk["chunk_id"] for chunk in chunks],
    documents=[chunk["text"] for chunk in chunks],
    metadatas=[
        {
            "chapter_number": chunk.get("chapter_number"),
            "chapter_title": chunk.get("chapter_title"),
            "token_count": chunk.get("token_count"),
        }
        for chunk in chunks
    ],
)

print(f"Stored {collection.count()} chunks.")



Stored 181 chunks.


Loading and Chunking exercies data

In [13]:
with open(EXERCISE_JSON_PATH, "r", encoding="utf-8") as f:
    raw_exercise_data = json.load(f)

exercise_items: List[ExerciseRecord] = [
    ExerciseRecord(**item) for item in raw_exercise_data["fitness"]
]

exercise_chunks = []

for idx, item in enumerate(exercise_items, start=1):
    searchable_text = "\n".join([
        f"Exercise Name: {item.exercise_name}",
        f"Activity Type: {item.type_of_activity}",
        f"Equipment: {item.type_of_equipment}",
        f"Body Part: {item.body_part}",
        f"Movement Type: {item.type}",
        f"Muscles Activated: {item.muscle_groups_activated}",
        f"Instructions: {item.instructions}",
    ])

    exercise_chunks.append({
        "id": f"exercise-{idx:04d}",
        "text": searchable_text,
        "metadata": {
            "exercise_name": item.exercise_name,
            "type_of_activity": item.type_of_activity,
            "type_of_equipment": item.type_of_equipment,
            "body_part": item.body_part,
            "type": item.type,
            "muscle_groups_activated": item.muscle_groups_activated,
            "source": "exercise_dataset",
        }
    })

print(f"Prepared {len(exercise_chunks)} exercise records.")
print(exercise_chunks[0]["id"])
print(exercise_chunks[0]["metadata"])
print(exercise_chunks[0]["text"])


Prepared 207 exercise records.
exercise-0001
{'exercise_name': 'Push-Ups', 'type_of_activity': 'Strength', 'type_of_equipment': 'Bodyweight', 'body_part': 'Upper Body', 'type': 'Push', 'muscle_groups_activated': 'Pectorals, Triceps, Deltoids', 'source': 'exercise_dataset'}
Exercise Name: Push-Ups
Activity Type: Strength
Equipment: Bodyweight
Body Part: Upper Body
Movement Type: Push
Muscles Activated: Pectorals, Triceps, Deltoids
Instructions: Start in a high plank position with your hands under your shoulders. Lower your body until your chest nearly touches the floor. Push back up to the starting position.


Storing exercies in DB

In [14]:

exercise_collection.upsert(
    ids=[item["id"] for item in exercise_chunks],
    documents=[item["text"] for item in exercise_chunks],
    metadatas=[item["metadata"] for item in exercise_chunks],
)

print(f"Stored {exercise_collection.count()}.")

/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:01<00:00, 75.3MiB/s]


Stored 207.


System prompt

In [32]:


system_prompt = """
You are a knowledgeable fitness coach and document-grounded assistant.

Your job is to answer the user's question using:
1. the provided document context
2. the conversation history when needed
3. tool results when a tool is used

Rules:
- Treat the provided document context as the primary source of truth for document-related questions.
- If the user explicitly asks for exercises, more exercises, exercise suggestions, workout ideas, drills, practice examples, or variations, you must call the retrieve_more_exercises tool.
- For mixed requests, first answer the document-based part using the provided context, then use the tool for the exercise-suggestion part when applicable.
- Do not invent facts, exercises, instructions, or recommendations that are not supported by the document context or tool results.
- If the answer cannot be found in the provided document context or tool results, respond exactly with:
The answer is not available in the provided document.
- Keep the answer concise, accurate, and directly relevant.
- Do not call the tool based only on previous conversation turns unless the current message clearly refers to that earlier exercise request.
"""



Retrive context

In [23]:
def search_context(collection_obj, query: str, top_k: int = TOP_K_RESULTS) -> str:
    results = collection_obj.query(
        query_texts=[query],
        n_results=top_k
    )

    docs = results.get("documents", [[]])[0]

    if not docs:
        return "No results found."

    return "\n\n---\n\n".join(docs)


def retrieve_more_exercises(query: str, top_k: int = 5) -> str:
    return search_context(exercise_collection, query, top_k)


def retrieve_information(prompt: str) -> str:
    book_context = search_context(collection, prompt, TOP_K_RESULTS)

    response = client.responses.create(
        model=MODEL,
        #conversation=conversation.id,
        instructions=system_prompt,
        input=f"Question:\n{prompt}\n\nDocument context:\n{book_context}",
        tools=[
            {
                "type": "function",
                "name": "retrieve_more_exercises",
                "description": (
                    "Use this only when the user explicitly asks for more exercises, "
                    "additional exercise suggestions, drills, practice examples, or workout ideas."
                ),
                "parameters": RetrieveExercisesArgs.model_json_schema()
            }
        ]
    )

    tool_outputs = []

    for item in response.output:
        if item.type == "function_call" and item.name == "retrieve_more_exercises":
            print("A tool was called")

            args = RetrieveExercisesArgs.model_validate_json(item.arguments)
            exercise_context = retrieve_more_exercises(
                args.query,
                args.top_k
            )

            tool_outputs.append({
                "type": "function_call_output",
                "call_id": item.call_id,
                "output": exercise_context
            })

    if not tool_outputs:
        return response.output_text

    follow_up = client.responses.create(
        model=MODEL,
        #conversation=conversation.id, #remove previous_response_id=response.id, if conversation is used
        previous_response_id=response.id,
        instructions=system_prompt,
        input=tool_outputs
    )
    return follow_up.output_text

Calling AI

In [17]:
def ask_ai(question: str):
    parsed = client.responses.parse(
        model=MODEL,
        input=[
            {
                "role": "developer",
                "content":
                """
                Extract the user's core question about a document and the desired response format.

                1. **format**
                  * Must be exactly one of:
                    * text
                    * audio

                Rules:
                * Use **audio** if the user asks to hear the answer or uses words such as:
                  tell, speak, speech, say, audio, voice, read aloud
                * Otherwise use **text**
                * If uncertain, default to **text**

                Return only the structured output matching the response schema.
                """
            },
            {
                "role": "user",
                "content": question
            }
        ],
        text_format=AskAIResponse
    )

    answer = retrieve_information(parsed.output_parsed.prompt)

    if parsed.output_parsed.format == "text":
        return answer

    if parsed.output_parsed.format == "audio":
        speech_file = client.audio.speech.create(
            model="gpt-4o-mini-tts",
            voice="alloy",
            input=answer
        )


        speech_file.write_to_file(PATH_TO_AUDIO_OUTPUT)
        return str(PATH_TO_AUDIO_OUTPUT)

Tests

In [21]:
print(ask_ai("What does the document say about programming with a purpose?"))

- Programming with a purpose means intentionally designing training to elicit specific adaptations, not leaving it to chance. It’s “results by design, not by coincidence,” using the SAID principle (Specific Adaptation to Imposed Demands).

- Training is framing appropriate stressors to drive predictable adaptations, with the body responding during recovery. The type of adaptation is verifiably controllable by choosing and manipulating the stressor.

- Programming is treated as problem solving. Because time is limited, you plan ahead (front-end work) with premade templates and then “platoon” or adjust them based on client goals and training age.

- The process is plan-driven: follow four steps—determine the goal, determine the starting point (training status), determine the time frame, plan backward and execute forward.

- Structure matters: use macrocycles, mesocycles, and microcycles. Reps, sets, loading, tempo, rest, and exercise selection (in that order of importance) are the main v

In [24]:
print(ask_ai("What does the document say about programming with a purpose?. Suggest some exercies."))

A tool was called
Here’s a concise take on programming with a purpose from the document, followed by related exercises.

Summary: programming with a purpose
- Core idea: Training is intentional stress application to drive specific adaptations (SAID principle) and occurs during recovery.
- Purposeful design: Choose stressors to elicit targeted adaptations; plan ahead rather than training by chance.
- Time constraints: Busy professionals use front-end planning and premade/templates programs, with flexible templates that are customized to individuals.
- Templates vs. rigidity: Templates are outlines (skeletons) that can be adjusted; don’t force the individual into a rigid template—fit the template to the client.
- Four steps of program design:
  1) Determine the goal
  2) Determine the starting point (training status)
  3) Determine the time frame
  4) Plan backward and execute forward
- Planning approach: Start with the end in mind (plan the macrocycle, mesocycles, and microcycles). Use 

In [25]:
print(ask_ai("Tell me how macrocycles, mesocycles, and microcycles work in a simple way."))

/content/last_answer.mp3


In [26]:
print(ask_ai("How can I track client progress over 8 weeks and know when to adjust the program?"))

Here’s a concise, document-grounded way to track progress over 8 weeks and know when to adjust the program.

What to track (three pillars)
- Look better / feel better / perform better
  - Look/fit: body composition changes, clothing fit
  - Feel: sleep quality, energy, mood, recovery, illness/symptoms
  - Perform: strength gains (loads/reps), skill tests relevant to goals (e.g., 1RM deadlift, mileage, vertical jump)
- Objective metrics
  - Strength/loads: note lifts and rep targets; track progress week to week
  - Body measures: body composition monthly; weight monthly (focus on fat loss changes, not just scale)
  - Performance tests: schedule at points (see cadence below)
- Readiness and consistency
  - Sleep hours and quality, daily energy/motivation
  - Training consistency (attendance, adherence to planned workouts)
  - Training readiness (use the readiness chart if you have one)

Cadence to use (based on the document)
- Every workout
  - Check: Are loads increasing or improving re

In [27]:
print(ask_ai("What are the best exercise progressions for squats and deadlifts for an intermediate client?"))

A tool was called
Here’s a concise, document-based roadmap for progressing squats and deadlifts for an intermediate client, followed by additional exercise options from the retrieve_more_exercises tool.

Document-based progressions (intermediate-friendly ladders)

- Squats
  - Asymmetrical (split stance)
    - Progression path (easiest to harder): assisted bottoms-up split squat → bodyweight split squat → goblet split squat → two-dumbbell split squat → goblet rear-foot-elevated split squat → two-dumbbell rear-foot-elevated split squat → goblet reverse lunge → two-dumbbell reverse lunge → two-dumbbell forward lunge or two-dumbbell walking lunge
    - Harder option: lateral lunge and cross-behind lunge
  - Symmetrical (parallel stance)
    - Progression path: assisted squat or bodyweight box squat → goblet squat → two-kettlebell front squat → barbell front squat
    - Harder option: barbell back squat
  - Single-leg stance
    - Progression path: bodyweight step-up → goblet step-up → gob

In [28]:
print(ask_ai("Suggest a 3-day fat-loss workout plan for a beginner with limited gym experience."))

A tool was called
Here is a clear 3-day fat-loss plan for a beginner with limited gym experience. It follows the template in the document (3 non-consecutive days, 2–3 sets of 8–12 reps for main lifts, 60–90 seconds rest, warm-up, cardio finish, and progression rules).

Day A – Full body (Push/Pull/Legs mix)
- Goblet squat or bodyweight squat – 2-3 x 8-12
- Dumbbell bench press or chest press machine – 2-3 x 8-12
- Seated row or dumbbell row – 2-3 x 8-12
- Lat pulldown or assisted pull-up – 2-3 x 8-12
- Dumbbell shoulder press or machine shoulder press – 2-3 x 8-12
- Hip hinge (Romanian deadlift with dumbbells) – 2-3 x 8-12
- Plank – 3 x 20-30 seconds
- Optional cardio finisher: 10 minutes at a moderate pace

Day B – Lower-body emphasis with core
- Leg press or goblet squat – 3 x 8-12
- Romanian deadlift or hip hinge (dumbbells) – 3 x 8-12
- Walking lunges or step-ups – 2-3 x 8-12 per leg
- Leg curl or hamstring curl machine – 2-3 x 10-15
- Calf raises – 2-3 x 12-15
- Glute bridge or hi

In [29]:
print(ask_ai("Speak a short summary of how to design a muscle-building program for someone training 4 days per week."))

/content/last_answer.mp3


In [30]:
print(ask_ai("How should I assess a client's training age before creating a strength program?"))

Here’s how to assess a client’s training age before designing a strength program, per the document:

1) Run a strategy session (about 1 hour) with four steps:
- Determine training goals.
- Gather health/medical history in detail.
- Gather training history and preferences, using the training-age rubric to qualify.
- Conduct a Functional Movement Screen (FMS) to establish baseline movement quality.

2) Qualify training age with the 6-question rubric (Figure 2.2):
- Score each question from 1 to 3, then sum the six answers.
- Questions cover:
  - Q1: Resistance training experience (structure and coaching history)
  - Q2: Last time trained consistently (current consistency)
  - Q3: General daily activity level
  - Q4: Athletic/competitive background
  - Q5: Current strength level (ability to perform key lifts with certain loads)
  - Q6: FMS baseline score (movement quality)
- Notes:
  - If someone hasn’t trained for more than 3 months, they receive a 1 for Q2 (detrained state).
  - Q5 is a